In [1]:
import polars as pl
import ast
import numpy as np
import time
import os
from tqdm import tqdm
from multiprocessing import Pool
import itertools
import matplotlib.pyplot as plt
import gc
from collections import Counter
import math
import keras
from scipy import sparse
from keras import models, layers
import keras
import glob
import gc

In [2]:
class CustomDataGenerator(keras.utils.Sequence):
    def __init__(self,
                src_data,
                batch_size, 
                dataset_type,
                **kwargs):
        super().__init__(**kwargs)
        self.src_data = src_data
        self.batch_size = batch_size
        assert dataset_type in ["train", "validation", "test"]
        self.dataset_type = dataset_type

        self.batch_map = []  # (dataset_idx, local_start_row)
        self.total_batches = 0
        self.list_dataset = []
        self.prepare_datasets()

    def prepare_datasets(self):
        self.batch_map = []

        dataset_type_folder = os.path.join(self.src_data, f"data_{self.dataset_type}")
        list_file_data = list(glob.glob(os.path.join(dataset_type_folder, '*.parquet')))
        for dataset_idx, file_data in enumerate(list_file_data):
            df = pl.scan_parquet(file_data)
            self.list_dataset.append(df)
            n_rows = df.select(pl.len()).collect().item()
            n_batches = math.ceil(n_rows / self.batch_size)
            for i in range(n_batches):
                self.batch_map.append((dataset_idx, i * self.batch_size))

        self.batch_map = tuple(self.batch_map)
        self.list_dataset = tuple(self.list_dataset)
        self.total_batches = len(self.batch_map)

    def __len__(self):
        return self.total_batches

    def normalize_dataset(self, dataset: pl.DataFrame):
        bool_cols = [col for col in dataset.columns if dataset[col].dtype == pl.Boolean]
        dataset = dataset.with_columns(
            [
                pl.col(bool_cols).cast(pl.Int8),  # Convert boolean columns to Int8
                (pl.col("player2_action_max_card") + 1) / 52,  # Normalize player2_action_max_card
                (pl.col("player3_action_max_card") + 1) / 52,  # Normalize player3_action_max_card
                (pl.col("action_max_card") + 1) / 52,  # Normalize action_max_card
            ]
        )
        return dataset

    def __getitem__(self, index):
        dataset_idx, start_row = self.batch_map[index]

        batch_df = self.list_dataset[dataset_idx].slice(start_row, self.batch_size).collect()
        batch_df = self.normalize_dataset(batch_df)
        np_data = batch_df.to_numpy()
        X = np_data[:, :-1]
        y = np_data[:, -1]
        return X, y

src_data = r"folderdata"
batch_size = 4096
data_train_generator = CustomDataGenerator(
    src_data=src_data,
    batch_size=batch_size,
    dataset_type="train")
data_validation_generator = CustomDataGenerator(
    src_data=src_data,
    batch_size=batch_size,
    dataset_type="validation")

In [ ]:
model = models.Sequential()
model.add(layers.Input(shape=(data_train_generator[0][0].shape[1],)))
model.add(layers.Dense(1024, kernel_initializer='he_normal'))
model.add(layers.BatchNormalization())
model.add(layers.PReLU(alpha_initializer=keras.initializers.Constant(value=0.25)))
model.add(layers.Dense(512, kernel_initializer='he_normal'))
model.add(layers.BatchNormalization())
model.add(layers.PReLU(alpha_initializer=keras.initializers.Constant(value=0.25)))
model.add(layers.Dense(256, kernel_initializer='he_normal'))
model.add(layers.BatchNormalization())
model.add(layers.PReLU(alpha_initializer=keras.initializers.Constant(value=0.25)))
model.add(layers.Dense(128, kernel_initializer='he_normal'))
model.add(layers.BatchNormalization())
model.add(layers.PReLU(alpha_initializer=keras.initializers.Constant(value=0.25)))
model.add(layers.Dense(64, kernel_initializer='he_normal'))
model.add(layers.BatchNormalization())
model.add(layers.PReLU(alpha_initializer=keras.initializers.Constant(value=0.25)))
model.add(layers.Dense(32, kernel_initializer='he_normal'))
model.add(layers.BatchNormalization())
model.add(layers.PReLU(alpha_initializer=keras.initializers.Constant(value=0.25)))
model.add(layers.Dense(16, kernel_initializer='he_normal'))
model.add(layers.BatchNormalization())
model.add(layers.PReLU(alpha_initializer=keras.initializers.Constant(value=0.25)))
model.add(layers.Dense(1, activation="sigmoid"))

model.summary()

In [ ]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model_checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath=r"TLMN_see_opponent_card_3_players.keras",
    monitor='val_accuracy',
    mode='max',
    save_best_only=True,
    verbose=1)

history = model.fit(data_train_generator, 
                    validation_data=data_validation_generator,
                    epochs=50)

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Mean Squared Error')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['mae'], label='Training MAE')
plt.plot(history.history['val_mae'], label='Validation MAE')
plt.title('Mean Absolute Error')
plt.xlabel('Epoch')
plt.ylabel('MAE')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
#Evaluate the model on the testing set
from keras.models import load_model

best_model = load_model(r"F:\Epoch_3_TLMN_see_opponent_card_20k_games_NVIDIA_Quadro_P2200.keras")
data_test_generator = CustomDataGenerator(
    src_data=r"F:\big_data\data_test", 
    dataset_type="test")
best_model.evaluate(data_test_generator)